# 13 · Room- and activity-fixed cluster strength (BV 2025.2)

Notebook 11 removed *shared scenes* (subject split); notebook 12 controlled *typical settings* statistically
(co-occurrence). Here the typical setting is held fixed directly, using the room and activity label of every
10-s BV clip (`2025.2/outputs/merged_contexts.csv`).

**Tests**
1. **Same room.** Category means from one room only (living room, kitchen, dining room, bedroom). If food still
   clusters with food when every crop comes from the living room, room context is not what drives it.
   Compared with all rooms pooled, on the same categories.
2. **Cross room.** Category i from room A vs category j from room B (symmetrized), so no pair shares a room.
3. **Activity.** The same two tests for activities (e.g. eating vs playing).
4. **vs THINGS** on each test's category set (THINGS has no rooms; it is the same matrix every time).

**Decisions**
- Crops: 2025.2 packed shards, CLIP image–label similarity ≥ 0.27, L2-normalized, subject 00270001 excluded
  (same as notebooks 11–12 and manuscript Stage 0). Frame `f` (1 fps) belongs to clip `f // 10`.
- Room and activity labels come from a video-language model run on 10-s clips, so they are noisy; frames in
  unlabeled clips are kept only in the pooled baseline.
- A category enters a room's (or activity's) matrix with ≥ `MIN_N` crops there. Every comparison is on one fixed
  category set; each matrix is z-scored across that set (manuscript convention).
- Measures as in notebooks 11–12: AUC = P(between-domain distance > within-domain distance) (scale-free;
  primary), Δ/SD, raw Δ. Differences: within-domain category bootstrap, self-pairs dropped, shared draws.
- Runs single-threaded at low priority; one shard pass (~25 min) builds a cache of per-(household, label,
  category) sums.

**Outputs:** `outputs/context_fixed/results/`, `figures/`; cache in `outputs/context_fixed/cache/` (git-ignored).

In [ ]:
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[_v] = '1'   # keep CPU use low (shared machine)
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

# Manuscript figure style (same as analysis/qwen-open-vocab/fig_style.py): Lato, bold axis labels and legends.
import matplotlib.axes, matplotlib.axis, matplotlib.figure
from matplotlib import font_manager
for _f in (Path.cwd().resolve().parents[1] / "talk-figures" / "unr-2026" / "fonts").glob("*.ttf"):
    font_manager.fontManager.addfont(str(_f))
plt.rcParams.update({
    "font.family": ["Lato", "DejaVu Sans", "sans-serif"], "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.labelweight": "bold", "figure.titlesize": 12, "figure.titleweight": "bold",
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#444444", "axes.labelcolor": "#222222",
    "text.color": "#222222", "xtick.color": "#444444", "ytick.color": "#444444"})
def _bold_legend(orig):
    if getattr(orig, "_bold", False):
        return orig
    def legend(self, *args, **kw):
        prop = dict(kw.pop("prop", {}) or {}); prop.setdefault("weight", "bold")
        prop.setdefault("size", kw.pop("fontsize", plt.rcParams["legend.fontsize"]))
        return orig(self, *args, prop=prop, **kw)
    legend._bold = True
    return legend
matplotlib.axes.Axes.legend = _bold_legend(matplotlib.axes.Axes.legend)
matplotlib.figure.Figure.legend = _bold_legend(matplotlib.figure.Figure.legend)
_tick_init = matplotlib.axis.Tick.__init__
if not getattr(_tick_init, "_bold", False):
    def _bold_tick(self, *a, **k):                       # bold tick labels (no rcParam for tick label weight)
        _tick_init(self, *a, **k); self.label1.set_fontweight("bold"); self.label2.set_fontweight("bold")
    _bold_tick._bold = True; matplotlib.axis.Tick.__init__ = _bold_tick
    _grid = matplotlib.axes.Axes.grid                    # no grid lines in any figure
    matplotlib.axes.Axes.grid = lambda self, visible=None, which="major", axis="both", **kw: _grid(self, False, which=which, axis=axis)
plt.rcParams["axes.grid"] = False
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [ ]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

MANUSCRIPT_SCRIPTS = MANUSCRIPT_DIR / "scripts"
BV_PACKED = Path("/ccn2a/dataset/babyview/2025.2/outputs/image_embeddings/babyview_full")
BV_CONTEXTS = Path("/ccn2a/dataset/babyview/2025.2/outputs/merged_contexts.csv")
STAGE0 = MANUSCRIPT_DIR / "exemplar_set_embeddings"
THINGS_RAW = str(STAGE0 / "valid129" / "things_{m}_exemplar_avg_raw_within_valid129.csv")
STAGE0_BV_RAW = str(STAGE0 / "2025.2" / "valid129" / "bv_{m}_exemplar_avg_raw_within_valid129.csv")

CF_OUT = OUT / "context_fixed"
CF_CACHE, CF_RESULTS, CF_FIGURES = CF_OUT / "cache", CF_OUT / "results", CF_OUT / "figures"
for d in (CF_CACHE, CF_RESULTS, CF_FIGURES):
    d.mkdir(parents=True, exist_ok=True)

## Parameters

In [ ]:
MIN_SIM = 0.27
EXCLUDE_BV_SUBJECTS = ("00270001",)
MIN_N = 20                                   # crops per category within a room / activity
ROOMS = ["living room", "kitchen", "dining room", "bedroom"]
ACTIVITIES = ["playing", "eating", "reading", "watching tv"]
N_BOOT = 1000
RECOMPUTE_CACHE = False
COLOR = {"bv": "#0D47A1", "things": "#90CAF9"}

## Helpers

In [ ]:
def load_cdi_semantic_map() -> dict[str, str]:
    """``uni_lemma`` → CDI semantic domain from ``data/cdi_words.csv`` (same as notebook 09)."""
    df = pd.read_csv(DATA_DIR / "cdi_words.csv", usecols=["uni_lemma", "category"])
    df["uni_lemma"] = df["uni_lemma"].astype(str).str.strip().str.lower()
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.drop_duplicates(subset=["uni_lemma"], keep="first").set_index("uni_lemma")["category"].to_dict()


CATS = sorted(l.strip().lower() for l in VALID129_TXT.read_text().splitlines() if l.strip())
CDI = load_cdi_semantic_map()


def zscore_rows(X, eps=1e-10):
    return (X - X.mean(axis=0)) / (X.std(axis=0) + eps)


def cosine_distance(A, B):
    A = A / np.linalg.norm(A, axis=1, keepdims=True); B = B / np.linalg.norm(B, axis=1, keepdims=True)
    return 1 - A @ B.T


def measures(R, sem, sel=None) -> dict:
    """AUC, Δ, Δ/SD over category pairs (bootstrap-safe: self-pairs dropped)."""
    sem = np.asarray(sem); sel = np.arange(len(sem)) if sel is None else sel
    s = sem[sel]; iu = np.triu_indices(len(sel), 1); keep = sel[iu[0]] != sel[iu[1]]
    v = R[np.ix_(sel, sel)][iu][keep]; same = (s[iu[0]] == s[iu[1]])[keep]; w, b = v[same], v[~same]
    rk = rankdata(np.concatenate([w, b]))
    auc = (rk[len(w):].sum() - len(b) * (len(b) + 1) / 2) / (len(w) * len(b))
    return {"auc": auc, "delta": b.mean() - w.mean(), "delta_sd": (b.mean() - w.mean()) / v.std()}


def domain_bootstrap(sem, n, seed):
    sem = np.asarray(sem); rng = np.random.default_rng(seed); doms = sorted(set(sem))
    return [np.concatenate([rng.choice(np.flatnonzero(sem == d), (sem == d).sum()) for d in doms]) for _ in range(n)]


def ci_p(x):
    lo, hi = np.percentile(x, [2.5, 97.5]); return lo, hi, float(2 * min((x <= 0).mean(), (x >= 0).mean()))

## Stage A · per-(household, room / activity, category) sums

Clip labels are matched to crops by video id and `frame_number // 10`.

In [ ]:
import sys
sys.path.insert(0, str(MANUSCRIPT_SCRIPTS))
from packed_crop_embeddings import (_row_mask, _vectors_for_mask, iter_shard_paths, l2_normalize_rows,
                                    subject_id_from_video_id)

CACHE = CF_CACHE / "bv_2025_2_household_label_category_sums.npz"


def load_clip_labels():
    mc = pd.read_csv(BV_CONTEXTS, usecols=["video_id", "Location", "Activity"], low_memory=False)
    mc["vid"] = mc["video_id"].astype(str).str.replace(r"_(\d+)\.mp4$", "", regex=True)
    mc["seg"] = mc["video_id"].astype(str).str.extract(r"_(\d+)\.mp4$")[0].astype(int)
    for c in ("Location", "Activity"):
        mc[c] = mc[c].astype(str).str.strip().str.lower()
    return mc


def build_cache():
    mc = load_clip_labels()
    locs = sorted(mc["Location"].unique()); acts = sorted(mc["Activity"].unique())
    lix = {v: i for i, v in enumerate(locs)}; aix = {v: i for i, v in enumerate(acts)}
    lab = {}
    for vid, g in mc.groupby("vid"):
        L = np.full(g["seg"].max() + 1, -1, int); A = L.copy()
        L[g["seg"].to_numpy()] = g["Location"].map(lix).to_numpy(); A[g["seg"].to_numpy()] = g["Activity"].map(aix).to_numpy()
        lab[vid] = (L, A)
    paths = [p for p in iter_shard_paths(BV_PACKED) if subject_id_from_video_id(p.stem) not in EXCLUDE_BV_SUBJECTS]
    hh = sorted({"H" + p.stem[1:5] for p in paths}); hix = {h: i for i, h in enumerate(hh)}
    dims = {"clip": 512, "dinov3": 768}; cix = {c: i for i, c in enumerate(CATS)}
    S = {(m, g): np.zeros((len(hh), n + 2, len(CATS), dims[m]), np.float32)          # last two: unlabeled, all
         for m in MODELS for g, n in (("loc", len(locs)), ("act", len(acts)))}
    N = {(m, g): np.zeros(S[(m, g)].shape[:3], np.int64) for (m, g) in S}
    for k, p in enumerate(paths):
        with np.load(p, allow_pickle=True) as z:
            sh = {kk: z[kk] for kk in z.files}
        h = hix["H" + p.stem[1:5]]; names = np.array([str(x).strip().lower() for x in sh["class_name"]])
        L, A = lab.get(p.stem, (np.array([], int), np.array([], int)))
        for m in MODELS:
            mask = _row_mask(sh, categories=set(CATS), min_sim=MIN_SIM, passed_only=False, model=m)
            if not mask.any():
                continue
            v = l2_normalize_rows(_vectors_for_mask(sh, mask, m)).astype(np.float32)
            c = np.array([cix[x] for x in names[mask]]); seg = sh["frame_number"][mask] // 10
            for g, lab_arr, n in (("loc", L, len(locs)), ("act", A, len(acts))):
                code_ = np.where(seg < len(lab_arr), lab_arr[np.minimum(seg, max(len(lab_arr) - 1, 0))] if len(lab_arr) else -1, -1)
                code_ = np.where(code_ < 0, n, code_)                       # unlabeled
                np.add.at(S[(m, g)][h], (code_, c), v); np.add.at(N[(m, g)][h], (code_, c), 1)
                np.add.at(S[(m, g)][h], (np.full_like(c, n + 1), c), v); np.add.at(N[(m, g)][h], (np.full_like(c, n + 1), c), 1)
        if k % 250 == 0:
            print(f"  {k}/{len(paths)}", flush=True)
    np.savez(CACHE, households=np.array(hh), locs=np.array(locs + ["(unlabeled)", "(all)"]),
             acts=np.array(acts + ["(unlabeled)", "(all)"]),
             **{f"{m}_{g}_sum": S[(m, g)] for (m, g) in S}, **{f"{m}_{g}_n": N[(m, g)] for (m, g) in N})


if RECOMPUTE_CACHE or not CACHE.exists():
    build_cache()
Z = np.load(CACHE)
LABELS = {"loc": list(Z["locs"]), "act": list(Z["acts"])}
print(len(Z["households"]), "households;", {g: len(v) - 2 for g, v in LABELS.items()}, "labels")

### Checks

The "(all)" slice must reproduce manuscript Stage 0 (2025.2), and labeled crops should cover most of the data.

In [ ]:
for m in MODELS:
    ref = pd.read_csv(STAGE0_BV_RAW.format(m=m)); ref["category"] = ref["category"].str.strip().str.lower()
    ref = ref.set_index("category").loc[CATS].filter(like="dim_").to_numpy()
    s, n = Z[f"{m}_loc_sum"][:, -1].sum(0), Z[f"{m}_loc_n"][:, -1].sum(0)
    err = np.abs(s / n[:, None] - ref).max(); print(f"{m:6s} max |all-rooms mean − Stage 0| = {err:.1e}"); assert err < 1e-5
n_all = Z["clip_loc_n"][:, -1].sum(); n_unl = Z["clip_loc_n"][:, -2].sum()
print(f"CLIP crops: {n_all:,}; in labeled clips: {1 - n_unl / n_all:.1%}")
crops_by_label = {g: pd.Series(Z[f"clip_{g}_n"].sum((0, 2))[:-2], index=LABELS[g][:-2]).sort_values(ascending=False) for g in LABELS}
display(pd.DataFrame({g: crops_by_label[g].head(8).index + " (" + crops_by_label[g].head(8).map("{:,}".format) + ")"
                      for g in crops_by_label}))

## Same-label and cross-label matrices

In [ ]:
def label_means(m, g, label):
    i = LABELS[g].index(label)
    s, n = Z[f"{m}_{g}_sum"][:, i].sum(0).astype(np.float64), Z[f"{m}_{g}_n"][:, i].sum(0)
    return s / np.maximum(n, 1)[:, None], n


def things_means(m):
    d = pd.read_csv(THINGS_RAW.format(m=m)); d["category"] = d["category"].str.strip().str.lower()
    return d.set_index("category").loc[CATS].filter(like="dim_").to_numpy(np.float64)


def run_group(g, labels):
    """Per label: same-label vs pooled vs THINGS on that label's categories; per label pair: cross-label vs same-label."""
    rows, pair_rows, rdms = [], [], {}
    for m in MODELS:
        pooled, n_all = label_means(m, g, "(all)"); things = things_means(m)
        per = {lab: label_means(m, g, lab) for lab in labels}
        for lab in labels:
            X, n = per[lab]; idx = np.flatnonzero(n >= MIN_N); sem = [CDI.get(CATS[i], "other") for i in idx]
            R = {"bv_same_label": cosine_distance(zscore_rows(X[idx]), zscore_rows(X[idx])),
                 "bv_pooled": cosine_distance(zscore_rows(pooled[idx]), zscore_rows(pooled[idx])),
                 "things": cosine_distance(zscore_rows(things[idx]), zscore_rows(things[idx]))}
            rdms[(m, g, lab)] = (idx, sem, R)
            for kind, r in R.items():
                rows.append({"model": m, "group": g, "label": lab, "n_categories": len(idx), "matrix": kind, **measures(r, sem)})
        for la, lb in combinations(labels, 2):
            (Xa, na), (Xb, nb) = per[la], per[lb]
            idx = np.flatnonzero((na >= MIN_N) & (nb >= MIN_N)); sem = [CDI.get(CATS[i], "other") for i in idx]
            A, B = zscore_rows(Xa[idx]), zscore_rows(Xb[idx])
            Rx = cosine_distance(A, B); Rx = (Rx + Rx.T) / 2
            Ra, Rb = cosine_distance(A, A), cosine_distance(B, B)
            Rt = cosine_distance(zscore_rows(things_means(m)[idx]), zscore_rows(things_means(m)[idx]))
            rdms[(m, g, f"{la} × {lb}")] = (idx, sem, {"bv_cross_label": Rx, "bv_same_label_a": Ra, "bv_same_label_b": Rb, "things": Rt})
            for kind, r in (("bv_cross_label", Rx), ("bv_same_label_a", Ra), ("bv_same_label_b", Rb), ("things", Rt)):
                pair_rows.append({"model": m, "group": g, "pair": f"{la} × {lb}", "n_categories": len(idx), "matrix": kind, **measures(r, sem)})
    return pd.DataFrame(rows), pd.DataFrame(pair_rows), rdms


same_room, cross_room, RDM_ROOM = run_group("loc", ROOMS)
same_act, cross_act, RDM_ACT = run_group("act", ACTIVITIES)
same = pd.concat([same_room, same_act]); cross = pd.concat([cross_room, cross_act])
same.to_csv(CF_RESULTS / "same_label_measures.csv", index=False); cross.to_csv(CF_RESULTS / "cross_label_measures.csv", index=False)
display(same.pivot_table(index=["model", "group", "label", "n_categories"], columns="matrix", values="auc").round(3))
display(cross.pivot_table(index=["model", "group", "pair", "n_categories"], columns="matrix", values="auc").round(3))

## Bootstrap tests

For each matrix set: BV same-label − BV pooled (does fixing the room change clustering?), BV same-label − THINGS
(does BV still cluster more once the room is fixed?), BV cross-label − mean of the two same-label matrices.

In [ ]:
def boot_rows(rdms, contrasts):
    out = []
    for key, (idx, sem, R) in rdms.items():
        draws = domain_bootstrap(sem, N_BOOT, 3)
        for a, b in contrasts:
            if a not in R or not all(x in R for x in (b if isinstance(b, tuple) else (b,))):
                continue
            def stat(sel, k):
                va = measures(R[a], sem, sel)[k]
                vb = np.mean([measures(R[x], sem, sel)[k] for x in (b if isinstance(b, tuple) else (b,))])
                return va - vb
            for k in ("auc", "delta_sd"):
                bt = np.array([stat(d, k) for d in draws]); lo, hi, p = ci_p(bt)
                out.append({"model": key[0], "group": key[1], "label": key[2], "n_categories": len(idx),
                            "contrast": f"{a} − {b if isinstance(b, str) else 'mean(' + ', '.join(b) + ')'}",
                            "metric": k, "diff": stat(None, k), "ci_lo": lo, "ci_hi": hi, "p_boot": p})
    return out


tests = pd.DataFrame(
    boot_rows({k: v for k, v in {**RDM_ROOM, **RDM_ACT}.items() if " × " not in k[2]},
              [("bv_same_label", "bv_pooled"), ("bv_same_label", "things"), ("bv_pooled", "things")])
    + boot_rows({k: v for k, v in {**RDM_ROOM, **RDM_ACT}.items() if " × " in k[2]},
                [("bv_cross_label", ("bv_same_label_a", "bv_same_label_b")), ("bv_cross_label", "things")]))
tests.to_csv(CF_RESULTS / "bootstrap_tests.csv", index=False)
display(tests[tests.metric == "auc"].round(3))

## Figure

In [ ]:
def fig_same_room():
    """AUC per room: BV same-room, BV pooled, THINGS (same categories); one panel per model."""
    d = same[same.group == "loc"]
    fig, axes = plt.subplots(1, len(MODELS), figsize=(9.6, 3.4), sharey=True, layout="constrained")
    style = {"bv_same_label": dict(marker="o", mfc=COLOR["bv"], label="BV, one room only"),
             "bv_pooled": dict(marker="o", mfc="white", label="BV, all rooms pooled"),
             "things": dict(marker="D", mfc=COLOR["things"], label="THINGS")}
    for ax, m in zip(axes, MODELS):
        dm = d[d.model == m]
        for y, room in enumerate(ROOMS):
            r = dm[dm.label == room].set_index("matrix")
            ax.plot([r.auc.min(), r.auc.max()], [y, y], color="0.8", lw=1.5, zorder=1)
            for kind, st in style.items():
                ax.plot([r.loc[kind, "auc"]], [y], ls="", ms=8, mec="0.2", zorder=3, **{k: v for k, v in st.items() if k != "label"})
        ax.set_yticks(range(len(ROOMS)), [f"{x} ({int(dm[dm.label == x].n_categories.iloc[0])})" for x in ROOMS])
        ax.set_ylim(len(ROOMS) - 0.5, -0.5); ax.set_title("CLIP" if m == "clip" else "DINOv3", fontsize=11)
        ax.set_xlabel("AUC: P(between > within distance)"); ax.grid(axis="x", color="0.9"); ax.set_axisbelow(True)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
    fig.legend(handles=[plt.Line2D([], [], ls="", ms=8, mec="0.2", **st) for st in style.values()],
               loc="outside lower center", ncol=3, frameon=False, fontsize=9)
    fig.suptitle("Domain clustering with the room held fixed (categories with ≥ %d crops in that room)" % MIN_N, fontsize=11)
    out = CF_FIGURES / "same_room_auc.png"; fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig)
    return out


show_images([fig_same_room()], width=820)

### Review checklist

- Does the "(all)" slice reproduce Stage 0, and what share of crops falls in labeled clips?
- Are label matrices compared on one fixed category set each (≥ `MIN_N` crops)?
- Same-room vs pooled: does fixing the room lower BV clustering (room context matters) or not?
- Same-room BV vs THINGS: does BV's advantage (DINOv3) persist with the room fixed?
- Cross-room vs same-room: pairs never share a room; does clustering hold?
- Labels are model-generated and noisy; rooms differ in which categories they contain (category sets differ by room).